## DL Hackathon Starter Colab
### Date: April 27, 2026
### DSBA+ICEF, HSE University

<small><font color=gray>Notebook authors: <a href="https://www.hse.ru/en/staff/aboldyrev" target="_blank">Alexey Boldyrev</a>, <a href="www.hse.ru/en/staff/mekarpov" target="_blank">Maksim Karpov</a>, <a href="https://www.hse.ru/en/staff/sara" target="_blank">Saraa Ali</a>, <a href="http://wiki.cs.hse.ru/Deep_Learning_DSBA_2025/2026" target="_blank">Stanislav Ryazanov</a>.

[**Instructions**](https://colab.research.google.com/drive/1owkYjuRGkx050LQnM3b3yTzd0Dr2XbeV) for running Colabs.

## Problem Description

**Motivation**: One of the most valuable sources of customer information is bank transaction data. In this set, there are many answers to the question: is it possible to predict the gender of a client using information about receipt and payment by bank card? And if so, what is the accuracy of such a prediction?

**Task**: Predict ROC AUC from the probability of gender (0|1) for each `cid` (customer ID) which is missing a gender in the file `gender.csv`.

<small>**CONSENT.** <mark>[ X ]</mark> We consent to sharing our Colab (after the Hackathon ends) with other students/instructors for educational purposes.

In [1]:
from google.colab import userdata

kaggle_token = 'KGAT_80e02d5555c148b63531253de8980235'

In [2]:
!pip install kaggle                                                 # upgrade kaggle package (to avoid a warning)
!mkdir -p ~/.kaggle                                      # .kaggle folder must contain kaggle.json for kaggle executable to properly authenticate you to Kaggle.com
!echo "$kaggle_token" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token                        # give only the owner full read/write access to kaggle.json
!kaggle config set -n competition -v hse-dl-hackathon-2026          # hackathon dataset
!kaggle competitions download -c hse-dl-hackathon-2026 >> log                                # download competition dataset as a zip file
!unzip -o *.zip >> log                                              # kaggle dataset is copied as a single file and needs to be unzipped
!kaggle competitions leaderboard -c hse-dl-hackathon-2026 --show                             # print public leaderboard

- competition is now set to: hse-dl-hackathon-2026
Next Page Token = CfDJ8CS0IeAoHcJGgSEc27rBbk5mgtaCFWf4jzQTyTpy-2azEHEJ8EBVslBhXMYdT5NTfqQ3OWUaYZx-URmZVAv1uuk
  teamId  teamName                     submissionDate              score    
--------  ---------------------------  --------------------------  -------  
15748694  BG                           2026-04-27 08:19:46.486000  0.88954  
15748538  F                            2026-04-27 08:42:03.653000  0.88941  
15748318  Maksim Rodikov               2026-04-27 08:21:31.946000  0.88852  
15748202  N Team                       2026-04-27 08:42:47.256000  0.88830  
15748268  AV - AnyaVova                2026-04-27 08:42:53.396000  0.88783  
15748158  AA                           2026-04-27 08:47:16.456000  0.88705  
15748321  AE                           2026-04-27 08:24:17.220000  0.88682  
15748186  AC                           2026-04-27 08:29:47.186000  0.88676  
15748529  BM                           2026-04-27 07:59:04.110000  0.

In [3]:
!pip install git+https://github.com/dllllb/pytorch-lifestream.git lightgbm

  Cloning https://github.com/dllllb/pytorch-lifestream.git to /tmp/pip-req-build-gv2weyns
  Running command git clone --filter=blob:none --quiet https://github.com/dllllb/pytorch-lifestream.git /tmp/pip-req-build-gv2weyns
  Resolved https://github.com/dllllb/pytorch-lifestream.git to commit 73a7a5490b98c468b666eaec332deae05e67f34d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
print(f'Device: {DEVICE}')

pd.set_option('display.max_rows', 100, 'display.max_columns', 100, 'display.max_colwidth', 100, 'display.precision', 2, 'display.max_rows', 4)

class Timer():
  def __init__(self, lim:'RunTimeLimit'=200): self.t0, self.lim, _ = time.time(), lim, print(f'⏳ started. You have {lim} sec. Good luck!')
  def ShowTime(self):
    msg = f'Runtime is {time.time()-self.t0:.0f} sec'
    print(f'\033[91m\033[1m' + msg + f' > {self.lim} sec limit!!!\033[0m' if (time.time()-self.t0-1) > self.lim else msg)

# Set all random numbers to ensure that your private LB score for Kaggle is reproducible with IPYNB file, which you submit via LMS.
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"Random seed set as {seed}")

Device: cuda


### Load data

In [5]:
gender = pd.read_csv('gender.csv', index_col='cid')
dfTrx  = pd.read_csv('trx.csv')

tY = gender.dropna()                # labelled CIDs (train)
vY = gender[gender.gender.isna()]   # unlabelled CIDs (test)
print(f'transactions={len(dfTrx):,}  train CIDs={len(tY):,}  test CIDs={len(vY):,}')

transactions=3,749,578  train CIDs=6,400  test CIDs=2,000


1. **trx.csv**, a table with chronological transaction history for ~15 months. Each `cid` can have unequal number of transactions.
    * `cid`, identifier of a bank's client; a non-negative integer
    * `dt`, a date of the transaction; an whole number starting from 0
    * `mcc`, mcc code of the transaction; a whole number; a foreign key for `mcc` in *_mcc.csv* table
    * `ttc`, transaction's type; a whole number; a foreign key for `ttc` in *_ttc.csv* table
    * `amt`, sum of the transactions in some monetary units, rounded to integer. Positive/Negative values are credits/debits to the client's account.
    * `tid`, ID of the register terminal (point of sale or POS) where the transaction was made.
1. **gender.csv**, a table with a numeric `cid` and `gender` (0|1) columns. Predict gender for the rows missing a gender value (i.e. first few thousands). Use the remaining gender values as target values in training your model.
1. **_mcc.csv**, a lookup table with the unique [Merchant Category Codes](https://en.wikipedia.org/wiki/Merchant_category_code) (MCC) and their descriptions in Russian. These might be helpful in locating similar categories using keywords or semantics (for example with sentence vectors).
1. **_ttc.csv**, a lookup table with unique transaction type codes and their verbal descriptions in Russian.

In [6]:
tmr = Timer() # runtime limit (in seconds). Add all of your code after the timer

⏳ started. You have 200 sec. Good luck!


<hr color=red>

<font size=5>⏳</font> <strong><font color=orange size=5>Your Code, Documentation, Ideas and Timer - All Start Here...</font></strong>

Students: Keep all your definitions, code, documentation **between** ⏳ symbols. Modifying any code outside of the timed playground incurs penalties.

## Preprocessing Pipeline

Explain elements of your preprocessing pipeline i.e. feature engineering, subsampling, clustering, dimensionality reduction, etc.

### 1. Build per-CID features

Two feature blocks: simple `amt` aggregates, and a normalized MCC count vector
(fraction of each CID's transactions falling into each MCC category).
MCC distribution is a strong gender signal: men and women shop at very different merchant types.

In [7]:
# 1. Установка и импорт библиотек (раскомментируйте установку ptsl и lightgbm, если их нет)
# !pip install git+https://github.com/dllllb/pytorch-lifestream.git lightgbm > /dev/null

import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from ptls.preprocessing import PandasDataPreprocessor
from ptls.nn import TrxEncoder, RnnSeqEncoder
from ptls.frames.coles import CoLESModule
from ptls.data_load.datasets import MemoryMapDataset
from ptls.data_load.iterable_processing import SeqLenFilter
from ptls.frames.coles.split_strategy import SampleSlices
from ptls.frames import PtlsDataModule

# --- ПАРАМЕТРЫ ---
COLES_EPOCHS = 3 # Увеличьте для лучшего качества (рекомендуется 60-150)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. ЗАГРУЗКА И ПРЕДОБРАБОТКА ДАННЫХ (ТАБЛИЧНЫЕ ПРИЗНАКИ)
# Преполагается, что dfTrx, tY (train) и vY (test) уже загружены 
print("Генерация табличных признаков из транзакций...")
tx = dfTrx.copy()
# dt - это уже дни. Вычисляем день недели
tx['dow'] = tx['dt'] % 7
tx['is_weekend'] = (tx['dow'] >= 5).astype(int)
tx['amount_abs'] = tx['amt'].abs()
tx['log_amount'] = np.log1p(tx['amount_abs'])

# Tier 1: MCC pivots
mcc_cnt = tx.groupby(['cid','mcc']).size().unstack(fill_value=0).add_prefix('mcc_cnt_')
mcc_sum = tx.groupby(['cid','mcc'])['amount_abs'].sum().unstack(fill_value=0).add_prefix('mcc_sum_')
mcc_share = mcc_sum.div(mcc_sum.sum(axis=1).replace(0,1), axis=0).add_prefix('share_')

# Tier 2: amount stats + debit/credit splits
amt = tx.groupby('cid')['amt'].agg(['sum','mean','std','median','min','max','count','skew'])
deb = tx[tx.amt < 0].groupby('cid')['amount_abs'].agg(['sum','mean','count']).add_prefix('deb_')
cre = tx[tx.amt > 0].groupby('cid')['amt'].agg(['sum','mean','count']).add_prefix('cre_')

# Определение вероятностной принадлежности МСС (взято из гайда)
FEMALE_MCC = [5977,5631,5621,5651,5641,5661,5691,5697,5944,5948,5992,7230,7297,7298]
MALE_MCC   = [5541,5542,5532,5533,5571,5734,5813,5921,5945,5993,7538,7941,7995,7997]
tx['fem'] = tx.mcc.isin(FEMALE_MCC)
tx['mal'] = tx.mcc.isin(MALE_MCC)
prior = tx.groupby('cid').agg(fem_share=('fem','mean'), mal_share=('mal','mean'))
prior['fem_minus_mal'] = prior['fem_share'] - prior['mal_share']

# Объединяем табличные признаки
df_tabular = mcc_cnt.join(mcc_sum).join(mcc_share).join(amt).join(deb).join(cre).join(prior).fillna(0)
print(f"Размерность табличных фичей: {df_tabular.shape}")

# 2. ПОДГОТОВКА ДАННЫХ ДЛЯ PYTORCH-LIFESTREAM (CoLES)
print("Обучение CoLES эмбеддингов с помощью ptls...")
preprocessor = PandasDataPreprocessor(
    col_id='cid',
    col_event_time='dt',
    event_time_transformation='none',
    cols_category=['mcc', 'ttc'],
    cols_numerical=['amt']
)

dataset_ptls = preprocessor.fit_transform(dfTrx)

trx_encoder_params = dict(
    embeddings_noise=0.003,
    numeric_values={'amt': 'log'},
    embeddings={
        'mcc': {'in': len(preprocessor.get_category_dictionary('mcc')), 'out': 24},
        'ttc': {'in': len(preprocessor.get_category_dictionary('ttc')), 'out': 12}
    }
)

seq_encoder = RnnSeqEncoder(
    trx_encoder=TrxEncoder(**trx_encoder_params),
    hidden_size=256, # В идеале 800-1024, убавлено для скорости
    type='gru',
)

coles_module = CoLESModule(
    seq_encoder=seq_encoder,
    optimizer_partial=lambda x: torch.optim.Adam(x, lr=0.005),
    weight_decay=0.0,
)

train_data = MemoryMapDataset(data=dataset_ptls)
train_dl = PtlsDataModule(
    train_data=train_data,
    train_num_workers=2,
    train_batch_size=128,
    split_strategy=SampleSlices(split_count=5, cnt_min=25, cnt_max=200)
)

trainer = pl.Trainer(
    max_epochs=COLES_EPOCHS, 
    accelerator="gpu" if DEVICE == 'cuda' else "cpu",
    devices=1,
    enable_progress_bar=False,
    logger=False
)

# Обучаем сеточку без учителя на всех транзакциях
trainer.fit(coles_module, train_dl)

# Инференс (получение эмбеддингов)
inference_dl = torch.utils.data.DataLoader(
    dataset=train_data,
    collate_fn=coles_module.trx_encoder.category_max_size,
    shuffle=False,
    batch_size=512,
    num_workers=2
)

coles_module.eval()
embeds = []
cids = []
with torch.no_grad():
    for batch in inference_dl:
        batch = inference_dl.collate_fn(batch)
        embeds.append(coles_module(batch).cpu().numpy())
        # Получение cid (ptls прячет структуру, но мы сохраняли порядок)
        
# Для экстракции правильных идентификаторов из ptls:
df_embeds = pd.DataFrame(torch.vstack([coles_module(batch) for batch in inference_dl]).numpy())
df_embeds['cid'] = [x['cid'] for x in dataset_ptls]
df_embeds = df_embeds.set_index('cid').add_prefix('ptls_')
print(f"Размерность CoLES фичей: {df_embeds.shape}")

# 3. ФОРМИРОВАНИЕ ФИНАЛЬНОГО ДАТАСЕТА И ОБУЧЕНИЕ LightGBM
X_full = df_tabular.join(df_embeds).fillna(0)

# Разделяем на train (где есть разметка) и test (где предсказываем)
X_train = X_full.loc[X_full.index.isin(tY.index)]
y_train = tY.loc[X_train.index, 'gender']

X_test = X_full.loc[X_full.index.isin(vY.index)]

print("Обучение ансамбля LightGBM...")
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'n_estimators': 500,
    'random_state': 42,
    'verbose': -1
}

# 5-fold Stratified CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    val_pred = model.predict_proba(X_va)[:, 1]
    oof_preds[val_idx] = val_pred
    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits
    print(f"Fold {fold} AUC: {roc_auc_score(y_va, val_pred):.4f}")

print(f"OOF ROC AUC: {roc_auc_score(y_train, oof_preds):.4f}")

# Формирование посылки
submission = pd.DataFrame({'cid': X_test.index, 'gender': test_preds}).set_index('cid')
submission.to_csv('submission.csv')
print("Успешно создано предсказание submission.csv!")

ModuleNotFoundError: No module named 'ptls.preprocessing.dask.dask_transformation'

## **References:**

* Remember to cite your sources here! At the least, your [Dive into Deep Learning](https://d2l.ai/) textbook should be cited. Google Scholar allows you to effortlessly copy/paste an APA citation format for books and publications. Also cite StackOverflow, package documentation, and other meaningful internet resources to help your peers learn from these (and to avoid plagiarism claims).

1. ...
1. ...
1. ...

<font color=green><h4><b>* LLM Documentation if used</b></h4></font>


1. Model and Platform Information  
   - The full name, version and operation mode of the model (e.g., Qwen (Qwen3.6-35B-A3B), Claude Opus 4.7, Gemini 3.1 Pro (Thinking mode)).  
   - A link to the service or platform used (e.g., https://chat.openai.com, https://claude.ai).  
   - If applicable, specify the application or interface (e.g., browser version with built‑in assistant, Telegram bot, IDE extension, etc.).

2. Interaction Record  
   - Provide all prompts and the model’s responses in chronological order (e.g., Prompt 1 – Response 1; Prompt 2 – Response 2, etc.).  
   - Ensure that the full conversation relevant to your submission is preserved and clearly formatted.

3. Reflection  
   - Briefly evaluate the quality and usefulness of the AI’s contribution.  
   - Explain what specific problem or part of the task the LLM helped you address.  
   - State how you verified or modified the output to ensure correctness and originality.

4. Usage Limitations  
   - Using an LLM to generate a complete solution is strictly prohibited.
   - Using LLM to generate documentation on the use of LLM is also prohibited.
   - The tool may be used only for assistance (e.g., drafting, brainstorming, clarifying concepts), not for full problem‑solving or code generation.

5. Academic Integrity  
   - Any submission incorporating LLM‑generated material without the documentation described above will be considered an academic integrity violation and may be treated as plagiarism.  
   - The teaching team reserves the right to determine whether a submission shows signs of unacknowledged AI assistance.


## 💡**Starter Ideas**

1. Richer hand-crafted features (signed-amt, log-magnitude, calendar, category-richness, TTC histogram, MCC×sign cross)
1. 5-fold stratified CV with Out-of-Fold (OOF) predictions
1. LightGBM/CatBoost as a complementary ensemble member
1. OOF-tuned ensemble averaging
1. GRU/Transformer sequence model with embedded (`mcc`, `ttc`, weekday, sign, log|`amt`|-bucket) tokens
1. Self-supervised pre-training and pseudo-labelling using the unlabelled test CIDs
1. Polishing (multi-seed, cosine LR, label smoothing, pos_weight)

<font size=5>⌛</font> <strong><font color=orange size=5>Do not exceed competition's runtime limit!</font></strong>

<hr color=red>

In [ ]:
tmr.ShowTime()    # measure Colab's runtime. Do not remove. Keep as the last cell in your notebook.

Runtime is 12 sec
